# OPT-1.3B LoRA Fine-Tuning
Fine-tuning **facebook/opt-1.3b** on Dolly-15k using LoRA. OPT is a base language model (not instruction-tuned), so fine-tuning teaches it instruction-following from scratch — providing a contrast to TinyLlama which is already instruction-tuned.

## Load Dataset


In [2]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

data = dataset["train"]

print(data[0])

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## Subset & Split
We use a 10,000-example subset (for practical training time on a single GPU) with an 80/20 train/test split. `seed=42` ensures reproducibility.

In [3]:
small = data.shuffle(seed=42).select(range(10000))

split = small.train_test_split(test_size=0.2)

train_data = split["train"]
test_data = split["test"]

## Format for Causal LM
We format each example into a single text string: `Instruction → Context (if present) → Answer`. This "completion-style" format is standard for causal language model fine-tuning — the model learns to generate the answer given the instruction prefix.

In [4]:
def format_example(ex):

    if ex["context"]:
        text = f"""Instruction: {ex['instruction']}

Context: {ex['context']}

Answer: {ex['response']}"""
    else:
        text = f"""Instruction: {ex['instruction']}

Answer: {ex['response']}"""

    return {"text": text}


train_data = train_data.map(format_example)
test_data = test_data.map(format_example)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
from transformers import pipeline 
ask_llm = pipeline(
    model= "facebook/opt-1.3b"
)

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [6]:
print(ask_llm('User: Explain AI models and applications in healthcare. Assistant:'))

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'User: Explain AI models and applications in healthcare. Assistant: Explain the AI models and applications in healthcare.\nAI systems are often used to make decisions in the healthcare industry. For example, AI systems may be used to make decisions about a patient’s risk of developing certain types of cancer. An AI model is a mathematical model that can be used to predict the probability that a particular patient will develop a certain type of cancer.'}]


## Load Model & Tokenizer

**Why OPT-1.3B?**
- A base (non-instruction-tuned) language model from Meta, providing a contrast to TinyLlama which is already instruction-tuned
- 1.3B parameters — comparable size to TinyLlama (1.1B), making the comparison fair
- Small enough to fine-tune on a single consumer GPU (< 5GB VRAM with LoRA)

We use `float16` precision to halve memory usage with minimal quality loss.

In [8]:
model_id = "facebook/opt-1.3b"

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)



In [9]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.float16
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Tokenization
- **`max_length=256`**: Covers ~95% of Dolly examples. Longer examples are truncated.
- **`padding="max_length"`**: Fixed-length sequences for efficient batching.
- **`labels = input_ids`**: For causal LM, the target is the input shifted by one — the Trainer handles the shift internally.

In [10]:
def tokenize(example):
    tok = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tok["labels"] = tok["input_ids"].copy()  
    return tok

In [11]:
train_tok = train_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## LoRA Configuration

| Parameter | Value | Why |
|-----------|-------|-----|
| `r=8` | Rank | Low rank keeps adapter small (~1M params). Higher rank = more capacity but more memory. |
| `lora_alpha=16` | Scaling | Alpha/r = 2 is a standard scaling factor. Controls how much the adapter affects the base model. |
| `target_modules=["q_proj","v_proj"]` | Attention layers | Adapting query and value projections is the most parameter-efficient choice for instruction-following. |
| `lora_dropout=0.05` | Regularization | Light dropout to prevent overfitting on 8K training examples. |

In [12]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,420,288,000 || trainable%: 0.1107


## Training Arguments

| Parameter | Value | Why |
|-----------|-------|-----|
| `per_device_train_batch_size=1` | Batch size | OPT-1.3B + LoRA fits in GPU memory with batch=1. |
| `num_train_epochs=1` | Epochs | Single pass over 8K examples. OPT is a base model so even 1 epoch shows clear improvement. |
| `learning_rate=0.001` | LR | Higher than full fine-tuning because LoRA only updates ~1M parameters. |
| `fp16=True` | Mixed precision | Halves memory, speeds up training on NVIDIA GPUs. |

In [13]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="lora-dolly",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    learning_rate=0.001,
    logging_steps=10,
    save_steps=50,
    fp16=True,
)


In [14]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok
)

In [15]:
trainer.train()

Step,Training Loss
10,5.267374
20,2.243317
30,1.563656
40,1.371076
50,1.417983
60,0.995799
70,1.374946
80,1.243768
90,1.104834
100,1.283703


TrainOutput(global_step=4000, training_loss=1.2167162035703658, metrics={'train_runtime': 2929.3945, 'train_samples_per_second': 2.731, 'train_steps_per_second': 1.365, 'total_flos': 1.6135772700672e+16, 'train_loss': 1.2167162035703658, 'epoch': 1.0})

In [17]:
import torch

def generate_answer(prompt, max_new_tokens=150):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # move to model device (cpu / gpu)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    output = tokenizer.decode(out[0], skip_special_tokens=True)

    # remove prompt from output
    answer = output[len(prompt):].strip()

    return answer

In [18]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [19]:
preds = []
refs = []

for ex in test_data.select(range(20)):

    prompt = f"""
Instruction: {ex['instruction']}
Context: {ex['context']}
Answer:
"""

    out = generate_answer(prompt)

    preds.append(out)
    refs.append(ex["response"])

In [22]:
print(bleu.compute(predictions=preds, references=[[r] for r in refs]))
print(rouge.compute(predictions=preds, references=refs))

{'bleu': 0.01738760687876998, 'precisions': [0.15380786460925833, 0.03167420814479638, 0.009141696292534281, 0.002052334530528476], 'brevity_penalty': 1.0, 'length_ratio': 1.6033519553072626, 'translation_length': 2009, 'reference_length': 1253}
{'rouge1': np.float64(0.2597396272054821), 'rouge2': np.float64(0.0877182190530556), 'rougeL': np.float64(0.1873184469523379), 'rougeLsum': np.float64(0.19370492793578842)}


## Save Adapter
We save only the LoRA adapter weights (~5MB), not the full 1.3B model. To reload, load the base OPT model and apply the adapter — this is done in `opt_eval.ipynb`.

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model.save_pretrained('./model')
tokenizer.save_pretrained('./model')

('./model/tokenizer_config.json', './model/tokenizer.json')

## Next: Full Evaluation
Evaluation is handled in `opt_eval.ipynb` — we load the saved adapter, run balanced sampling, and compare metrics against DeepSeek and TinyLlama.